# Adversarial validation

In Assignment 3 our CV score was 0.89, but the public leaderboard gave 0.59 for the same model. One likely
reason is that the training sentences don't look like the test posts. This notebook checks that directly.

We mix training sentences (label 0) with test posts (label 1) and train a classifier to tell them apart.
Then we measure ROC AUC on texts it didn't train on. AUC is the chance that a random test post gets a
higher score than a random training sentence: 0.5 means the model can't tell the two sets apart, 1.0
means it always can.

The assignment asks to use the same model as the main task, so the classifier is `xlm-roberta-base` with
a classification head on top.

Training and preprocessing now live in `evaluation/domain_classifier.py`, shared with the
synthetic-data runner. Saved outputs below are historical results from the original run;
this refactor does not recompute them. The new grouped 50-post experiment is reported in
`generation_pilot/AUC_REPORT.md` and uses different sample sizes and splits.


In [1]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/koalla05/deep-learning-assignment4.git"

if "google.colab" in sys.modules and not Path("data").exists():
    subprocess.run(["git", "clone", "-q", REPO_URL], check=True)
    os.chdir("deep-learning-assignment4")

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer

from evaluation.domain_classifier import (
    ClassifierConfig,
    mask_and_keep_long as preprocess_controlled,
    train_and_score as run_classifier,
)

MODEL = "FacebookAI/xlm-roberta-base"
SEED = 42
MASKED_TOKENS = 32


## Data

Most test posts are in Ukrainian, so we sample the training sentences with the same language mix.
Otherwise the model could just learn the language.

In [3]:
cv = pd.read_parquet("data/cv_rows.parquet")
test = pd.read_csv("data/test.csv")

has_uk = test.text.str.contains("[іїєґІЇЄҐ]")
has_ru = test.text.str.contains("[ыэъёЫЭЪЁ]")
print(
    f"test posts: {len(test)}, Ukrainian letters: {has_uk.sum()}, Russian letters: {has_ru.sum()}, "
    f"both: {(has_uk & has_ru).sum()}, neither: {(~has_uk & ~has_ru).sum()}"
)

test posts: 477, Ukrainian letters: 428, Russian letters: 34, both: 1, neither: 16


We take twice as many sentences as posts, 90% Ukrainian and 10% Russian. Only one sentence per group,
so near-copies can't end up on both sides of the split. Then 80% of the texts train the classifier and
20% are held out for the AUC.

In [4]:
def sample_sentences(source, n):
    rows = cv[cv.source == source].drop_duplicates("group_id").drop_duplicates("text")
    return rows.sample(n, random_state=SEED).text


n_sentences = 2 * len(test)
n_uk = round(0.9 * n_sentences)
sentences = pd.concat([sample_sentences("uk", n_uk), sample_sentences("ru", n_sentences - n_uk)])

data = pd.DataFrame({
    "text": pd.concat([sentences, test.text], ignore_index=True),
    "label": [0] * len(sentences) + [1] * len(test),
})
fit, holdout = train_test_split(data, test_size=0.2, stratify=data.label, random_state=SEED)
print(f"train: {len(fit)}, held out: {len(holdout)}, texts in both: {len(set(fit.text) & set(holdout.text))}")

train: 1144, held out: 287, texts in both: 0


## Training

Both runs use the same settings, with no tuning: learning rate 2e-5, 2 epochs, batch size 16, seed 42.

In [5]:
tokenizer = AutoTokenizer.from_pretrained(MODEL)
config = ClassifierConfig(model=MODEL, seed=SEED, eval_batch_size=64)


def train_and_score(name, fit, holdout, max_length):
    return run_classifier(
        name, fit, holdout, max_length,
        tokenizer=tokenizer, config=config,
    )["auc"]


## Two runs

Run 1 uses the raw text, cut to 256 tokens.

Run 2 tries to hide format and length. Links, emoji and line breaks become spaces (the same `mask_cues`
as in Assignment 3). After that we keep only texts with at least 32 tokens and cut all of them to exactly
32, so every input has the same length. Just cutting to 32 isn't enough: most training sentences are
shorter than that and most posts are longer, so the length would still give the answer away.

In [6]:
def mask_and_keep_long(frame):
    return preprocess_controlled(frame, tokenizer, MASKED_TOKENS)


fit_masked, holdout_masked = mask_and_keep_long(fit), mask_and_keep_long(holdout)
print("texts with at least 32 tokens after masking (0 = sentences, 1 = posts)")
print(pd.DataFrame({
    "train": fit_masked.label.value_counts(),
    "train, all": fit.label.value_counts(),
    "held out": holdout_masked.label.value_counts(),
    "held out, all": holdout.label.value_counts(),
}))


texts with at least 32 tokens after masking (0 = sentences, 1 = posts)
       train  train, all  held out  held out, all
label                                            
0        285         763        76            191
1        304         381        74             96


In [7]:
pd.DataFrame([
    {"run": "1: raw text, 256 tokens", "auc": train_and_score("raw", fit, holdout, 256)},
    {"run": "2: masked, exactly 32 tokens", "auc": train_and_score("masked", fit_masked, holdout_masked, MASKED_TOKENS)},
]).round(3)

{'loss': '0.6734', 'grad_norm': '2.434', 'learning_rate': '1.875e-05', 'epoch': '0.1389'}


{'loss': '0.5756', 'grad_norm': '5.908', 'learning_rate': '1.736e-05', 'epoch': '0.2778'}


{'loss': '0.4808', 'grad_norm': '10.17', 'learning_rate': '1.597e-05', 'epoch': '0.4167'}


{'loss': '0.3532', 'grad_norm': '7.6', 'learning_rate': '1.458e-05', 'epoch': '0.5556'}


{'loss': '0.2426', 'grad_norm': '7.477', 'learning_rate': '1.319e-05', 'epoch': '0.6944'}


{'loss': '0.2049', 'grad_norm': '12.43', 'learning_rate': '1.181e-05', 'epoch': '0.8333'}


... (16 outputs left out)

{'loss': '0.5044', 'grad_norm': '13.58', 'learning_rate': '1.351e-06', 'epoch': '1.892'}


{'train_runtime': '12.57', 'train_samples_per_second': '93.68', 'train_steps_per_second': '5.885', 'train_loss': '0.6135', 'epoch': '2'}


{'eval_loss': '0.4229', 'eval_auc': '0.9716', 'eval_runtime': '0.2822', 'eval_samples_per_second': '531.6', 'eval_steps_per_second': '10.63', 'epoch': '2'}


,run,auc
0,"1: raw text, 256 tokens",0.992
1,"2: masked, exactly 32 tokens",0.972


For comparison, the word-level BiLSTM from Assignment 3 got AUC 0.94 on raw Ukrainian sentences and 0.84
with format and length controlled.

## What this means

Both AUCs are far above 0.5. On raw text the classifier separates sentences from posts almost perfectly
(0.99). With links, emoji, line breaks and length hidden it still gets 0.97 on 150 held-out texts, so the
difference is in the content itself, not only in the format. XLM-RoBERTa separates the two sets even
better than the BiLSTM did in Assignment 3.

So the training sentences are a poor stand-in for the test posts, and a high CV score on them says little
about the leaderboard. This fits what we saw in Assignment 3: CV around 0.89 against 0.59 on the
leaderboard, and no rank agreement between the two (see `eda.ipynb`). That's why we report the
leaderboard score next to CV for every experiment.

One limit: this check only sees differences in the text. It can't tell whether the test posts were
labeled by different rules.